In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!nvidia-smi
!pip -q install ultralytics
import torch, ultralytics, sys
print(ultralytics.__version__, sys.version.split()[0])
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Mon Nov 16 08:55:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!mkdir -p /content/data
!for c in United_States Japan Czech India; do unzip -q /content/drive/MyDrive/road-damage/$c.zip -d /content/data; done
!for c in United_States Japan Czech India; do echo $c; for s in train val test; do echo -n "$s: "; ls /content/data/$c/images/$s | wc -l; done; done

replace /content/data/United_States/labels/val/United_States_000005.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace /content/data/Japan/labels/val/Japan_000010.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C
United_States
train: 3323
val: 476
test: 961
Japan
train: 7257
val: 1040
test: 2102
Czech
train: 1588
val: 262
test: 566
India
train: 4457
val: 678
test: 1542


In [ ]:
import yaml

def make_yaml(c):
    d = {"path": f"/content/data/{c}",
         "train": "images/train", "val": "images/val", "test": "images/test",
         "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"}}
    p = f"/content/{c}.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(d, f)
    return p

from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data=make_yaml("Czech"), epochs=3, imgsz=640, batch=16,
            workers=2, seed=0, project="/content/runs", name="czech_smoke", plots=False)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Czech.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=czech_smoke, nbs=64, nms=None, opset=None, optimize=False, optimizer=a

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d625a14da70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
m = model.val(data=make_yaml("Czech"), split="test", imgsz=640, plots=False)
print("test mAP50:", round(m.box.map50, 4), " mAP50-95:", round(m.box.map, 4))
print("per-class mAP50:", dict(zip(["D00", "D10", "D20", "D40"], [round(x, 4) for x in m.box.ap50])))

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2376.2±868.7 MB/s, size: 130.1 KB)
val: Scanning /content/data/Czech/labels/test... 566 images, 349 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 566/566 2.4Kit/s 0.2s
val: New cache created: /content/data/Czech/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 6.0it/s 6.0s
                   all        566        354      0.792     0.0406     0.0301    0.00756
                   D00        151        191      0.167      0.162     0.0903     0.0232
                   D10         65         89          1          0     0.0286     0.0065
                   D20         20         25          1          0    0.00153   0.000512
                   D40         39         49          1          0        

In [ ]:
import yaml, shutil, time
from ultralytics import YOLO

COUNTRIES = ["Czech", "India", "Japan", "United_States"]
NAMES = ["D00", "D10", "D20", "D40"]
CFG = dict(imgsz=640, batch=16, workers=2, seed=0, epochs=50,
           optimizer="AdamW", lr0=0.001, patience=50, plots=False)

def make_yaml(c):
    d = {"path": f"/content/data/{c}",
         "train": "images/train", "val": "images/val", "test": "images/test",
         "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"}}
    p = f"/content/{c}.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(d, f)
    return p

def run(train_c, tag):
    t0 = time.time()
    model = YOLO("yolo11n.pt")
    model.train(data=make_yaml(train_c), project="/content/runs", name=tag, exist_ok=True, **CFG)
    print(f"\nTRAIN TIME: {(time.time() - t0) / 60:.1f} min")
    shutil.copytree(f"/content/runs/{tag}", f"/content/drive/MyDrive/road-damage/runs/{tag}", dirs_exist_ok=True)
    best = YOLO(f"/content/runs/{tag}/weights/best.pt")
    print("\nRESULTS (test split of each country)")
    for t in COUNTRIES:
        m = best.val(data=make_yaml(t), split="test", imgsz=640, plots=False, verbose=False)
        ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist()))
        print(f"{train_c} -> {t}: mAP50 {m.box.map50:.4f}  mAP50-95 {m.box.map:.4f}  | AP50 "
              + "  ".join(f"{NAMES[i]} {ap.get(i, float('nan')):.3f}" for i in range(4)))

run("Czech", "czech_full_s0")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Czech.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=czech_full_s0, nbs=64, nms=None, opset=None, optimize=False, optimize

In [ ]:
import yaml, shutil, time, random, csv
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

DRIVE = Path("/content/drive/MyDrive/road-damage")
RESULTS = DRIVE / "results.csv"
COUNTRIES = ["Czech", "India", "Japan", "United_States"]
CFG = dict(imgsz=640, batch=16, workers=2, seed=0, epochs=50,
           optimizer="AdamW", lr0=0.001, patience=50, plots=False)
N_SUB = 1588

def make_yaml(c, sub=False):
    train = "images/train"
    if sub:
        files = sorted(Path(f"/content/data/{c}/images/train").glob("*.jpg"))
        random.Random(0).shuffle(files)
        lst = Path(f"/content/data/{c}/train_sub{N_SUB}.txt")
        lst.write_text("\n".join(str(f) for f in files[:N_SUB]))
        train = str(lst)
    d = {"path": f"/content/data/{c}", "train": train, "val": "images/val",
         "test": "images/test", "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"}}
    p = f"/content/{c}{'_sub' if sub else ''}.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(d, f)
    return p

def done(train_c, setting):
    if not RESULTS.exists():
        return False
    df = pd.read_csv(RESULTS)
    return ((df.train == train_c) & (df.setting == setting)).sum() == 4

def log_rows(train_c, setting, weights, minutes):
    best = YOLO(weights)
    rows = []
    for t in COUNTRIES:
        m = best.val(data=make_yaml(t), split="test", imgsz=640, plots=False, verbose=False)
        ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist()))
        rows.append([train_c, setting, t, round(m.box.map50, 4), round(m.box.map, 4)]
                    + [round(ap.get(i, float("nan")), 4) for i in range(4)]
                    + [minutes, CFG["epochs"], CFG["seed"]])
    new = not RESULTS.exists()
    with open(RESULTS, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["train", "setting", "test", "map50", "map50_95",
                        "D00", "D10", "D20", "D40", "train_min", "epochs", "seed"])
        w.writerows(rows)

def train_and_log(train_c, setting):
    if done(train_c, setting):
        print("skip", train_c, setting)
        return
    sub = setting.startswith("equal")
    tag = f"{train_c.lower()}_{setting}_s0"
    t0 = time.time()
    YOLO("yolo11n.pt").train(data=make_yaml(train_c, sub), project="/content/runs",
                             name=tag, exist_ok=True, **CFG)
    minutes = round((time.time() - t0) / 60, 1)
    shutil.copytree(f"/content/runs/{tag}", DRIVE / "runs" / tag, dirs_exist_ok=True)
    log_rows(train_c, setting, f"/content/runs/{tag}/weights/best.pt", minutes)

# Czech baseline was already trained: log it from the weights saved on Drive
if not done("Czech", "full"):
    log_rows("Czech", "full", str(DRIVE / "runs" / "czech_full_s0" / "weights" / "best.pt"), 23.2)

for c in ["India", "Japan", "United_States"]:
    train_and_log(c, "equal1588")

print(pd.read_csv(RESULTS).to_string())

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2122.2±436.7 MB/s, size: 129.2 KB)
val: Scanning /content/data/Czech/labels/test.cache... 566 images, 349 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 566/566 87.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 5.0it/s 7.3s
                   all        566        354      0.238       0.31       0.24     0.0818
Speed: 1.8ms preprocess, 4.2ms inference, 0.0ms loss, 2.6ms postprocess per image
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3204.9±640.6 MB/s, size: 113.1 KB)
val: Scanning /content/data/India/labels/test.cache... 1542 im

In [ ]:
import yaml, shutil, time, random, csv
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from IPython.display import clear_output

DRIVE = Path("/content/drive/MyDrive/road-damage")
RESULTS = DRIVE / "results.csv"
COUNTRIES = ["Czech", "India", "Japan", "United_States"]
CFG = dict(imgsz=640, batch=16, workers=2, seed=0, epochs=50,
           optimizer="AdamW", lr0=0.001, patience=50, plots=False)
N_SUB = 1588

def make_yaml(c, sub=False):
    train = "images/train"
    if sub:
        files = sorted(Path(f"/content/data/{c}/images/train").glob("*.jpg"))
        random.Random(0).shuffle(files)
        lst = Path(f"/content/data/{c}/train_sub{N_SUB}.txt")
        lst.write_text("\n".join(str(f) for f in files[:N_SUB]))
        train = str(lst)
    d = {"path": f"/content/data/{c}", "train": train, "val": "images/val",
         "test": "images/test", "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"}}
    p = f"/content/{c}{'_sub' if sub else ''}.yaml"
    with open(p, "w") as f:
        yaml.safe_dump(d, f)
    return p

def done(train_c, setting):
    if not RESULTS.exists():
        return False
    df = pd.read_csv(RESULTS)
    return ((df.train == train_c) & (df.setting == setting)).sum() == 4

def log_rows(train_c, setting, weights, minutes):
    best = YOLO(weights)
    rows = []
    for t in COUNTRIES:
        m = best.val(data=make_yaml(t), split="test", imgsz=640, plots=False, verbose=False)
        ap = dict(zip(m.box.ap_class_index.tolist(), m.box.ap50.tolist()))
        rows.append([train_c, setting, t, round(m.box.map50, 4), round(m.box.map, 4)]
                    + [round(ap.get(i, float("nan")), 4) for i in range(4)]
                    + [minutes, CFG["epochs"], CFG["seed"]])
    new = not RESULTS.exists()
    with open(RESULTS, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["train", "setting", "test", "map50", "map50_95",
                        "D00", "D10", "D20", "D40", "train_min", "epochs", "seed"])
        w.writerows(rows)

def train_and_log(train_c, setting):
    if done(train_c, setting):
        print("skip", train_c, setting)
        return
    sub = setting.startswith("equal")
    tag = f"{train_c.lower()}_{setting}_s0"
    t0 = time.time()
    YOLO("yolo11n.pt").train(data=make_yaml(train_c, sub), project="/content/runs",
                             name=tag, exist_ok=True, **CFG)
    minutes = round((time.time() - t0) / 60, 1)
    shutil.copytree(f"/content/runs/{tag}", DRIVE / "runs" / tag, dirs_exist_ok=True)
    log_rows(train_c, setting, f"/content/runs/{tag}/weights/best.pt", minutes)

for c in ["United_States", "India", "Japan"]:
    train_and_log(c, "full")
    clear_output(wait=True)
    print("finished:", c)
    print(pd.read_csv(RESULTS).query("setting == 'full'")[["train", "test", "map50", "map50_95", "train_min"]].to_string())

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
!mkdir -p /content/data
!for c in United_States Japan Czech India; do [ -d /content/data/$c ] || unzip -q /content/drive/MyDrive/road-damage/$c.zip -d /content/data; done
!nvidia-smi --query-gpu=name --format=csv,noheader
!for c in United_States Japan Czech India; do echo -n "$c train: "; ls /content/data/$c/images/train | wc -l; done

Mounted at /content/drive
Tesla T4
United_States train: 3323
United_States train: 3323
United_States train: 3323
United_States train: 3323
